# Real data and real models

Works on `<repo>/data/` and `<repo>/prompts/`. Needs `data/kuki_ru_aggregate.jsonl` and
`data/kuki_tr_aggregate.jsonl` (not in git). Loading and splitting use the same scripts as
`test_pipeline.ipynb`. Always run in a **fresh kernel**. Order: check data -> prepare -> check prompts -> smoke test.


In [ ]:
import os
import sys

from dotenv import load_dotenv

load_dotenv()  # e.g. Hugging Face token / cache location
if "config" in sys.modules:
    raise RuntimeError("config is already imported in this kernel: restart the kernel and run from the top")
os.environ.pop("KUKI_TEST", None)  # make sure test mode is off
import config

os.chdir(config.ROOT)  # so that %run finds the scripts
missing = [str(p) for p in config.AGGREGATES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"copy the aggregate files here: {missing}")


## 1. Structural check
Expected (dataset description): RU 207 articles (1/2/3 annotators: 139/42/26), TR 230 (151/44/35);
users RU 8/4/10, TR 11/7/6; no unknown sources.


In [ ]:
import json
from collections import Counter

from llm_io import read_jsonl


def inspect_aggregate(lang):
    """Structural check of an aggregate file: counts, users, span layers."""
    recs = read_jsonl(config.AGGREGATES[lang])
    anns = [a for r in recs for a in r["annotations"].values()]
    print(f"{config.AGGREGATES[lang].name}: {len(recs)} articles, {len(anns)} annotations")
    print("  articles by number of annotators:", dict(sorted(Counter(r["n_annotators"] for r in recs).items())))
    print("  users:", dict(Counter(u for r in recs for u in r["annotations"])), "expected:", config.ANNOTATOR_IDS[lang])
    print("  span layers:", dict(Counter(s["layer"] for a in anns for s in a["spans"])))
    print("  sources:", dict(Counter(r["source"] for r in recs)), "unknown:", {r["source"] for r in recs} - set(config.SOURCES))


for lang in config.AGGREGATES:
    inspect_aggregate(lang)


## 2. Data preparation and splits


In [ ]:
%run prep_00_load_data.py

In [ ]:
%run prep_01_make_splits.py

## 3. Prompt check
What the model sees for one paragraph-with-context call in the native prompt language.

In [ ]:
import llm_io

article = llm_io.load_split("dev")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=1):
    print(f"--- {message['role']} ---\n{message['content'][:1500]}\n")

## 4. Smoke test with a real model
Loads the model with transformers and runs a few constrained calls on one dev article.
Check: valid JSON, token counts, seconds per call (x number of calls = runtime of a stage).

In [ ]:
import time

from tqdm.auto import tqdm

MODEL = "olmo3-7b"

start = time.time()
backend = llm_io.load_model(MODEL)
print(f"model loaded in {time.time() - start:.0f} s, dtype {backend['model'].dtype}")

jobs = llm_io.make_jobs("smoke", [article], MODEL, ["L1", "L2", "L3"], ["doc"], ["en"])
for job in tqdm(jobs, desc="smoke test"):
    record = llm_io.call_llm(backend, job)
    tqdm.write(f"{job['layer']} {record['parsed']} | tokens in/out: {record['tokens_in']}/{record['tokens_out']} "
               f"| {record['seconds']} s | error: {record['error']}")